# Study 813 — Maximum-Drawdown Anomaly — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer (both directions), and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3895, 'spread_bps': -4.35, 't_nw': -2.36, 't_1s': -2.34, 'lo_bps': 5.34, 'hi_bps': 9.68, 'welch_t': -1.5, 'gross_sharpe': -0.59, 'placebo_obs': -4.35, 'placebo_mean': 0.046, 'placebo_sd': 1.049, 'placebo_p': 1.0, 'placebo_sigma_left': 4.19, 'placebo_draws': 1000, 'era_early_bps': -2.41, 'era_early_t': -1.1, 'era_early_n': 1761, 'era_late_bps': -5.94, 'era_late_t': -2.09, 'era_late_n': 2134, 'timer_1_gross': -4.35, 'timer_1_cost': 2.14, 'timer_1_net': -6.48, 'timer_1_t': -3.49, 'timer_5_gross': -4.35, 'timer_5_cost': 10.14, 'timer_5_net': -14.48, 'timer_5_t': -7.79, 'flip_1_net': 2.21, 'flip_1_t': 1.19, 'flip_5_net': -5.79, 'flip_5_t': -3.11, 'null_mean_t': -0.53, 'null_sd_t': 1.07, 'null_fire': 2, 'planted_t': 9.03, 'planted_welch': 9.18, 'fingerprint': '357fd262912f'}

## The headline — long-calm / short-distressed spread

Daily equal-weight bottom-30% (shallow drawdown) minus top-30% (deep drawdown) spread. `spread = calm - distressed`; a negative value means the distressed names out-earned (a rebound).

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : calm {R['lo_bps']:+.2f} vs distressed {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (calm-minus-distressed, before cost)")

spread        : -4.35 bps/day  NW(10) t = -2.36  one-sample t = -2.34
books         : calm +5.34 vs distressed +9.68 bps (Welch t = -1.50)
gross Sharpe  : -0.59 (calm-minus-distressed, before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f}")
print(f"observed sits ~{R['placebo_sigma_left']:.2f} sigma into the LEFT tail -> the (reversal) spread is not a lucky sort")

observed -4.35 bps vs placebo mean +0.046 (sd 1.049) -> right-tail p = 1.00000
observed sits ~4.19 sigma into the LEFT tail -> the (reversal) spread is not a lucky sort


## Robustness — two eras (split 2018-01-01)

The decisive honesty check: does the rebound hold across time?

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}  <- NOT significant")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}  <- carries the full-sample result")

2010-2017 (n=1761): -2.41 bps  NW t = -1.10  <- NOT significant
2018-2026 (n=2134): -5.94 bps  NW t = -2.09  <- carries the full-sample result


## The timer — can you get paid for it (either direction)?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
print('specified book (long calm / short distressed):')
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"  {tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")
print('sign-flipped REBOUND book (long distressed / short calm):')
for tag,n,t in [('1 bp',R['flip_1_net'],R['flip_1_t']),('5 bps',R['flip_5_net'],R['flip_5_t'])]:
    print(f"  {tag:>5} one-way: net {n:+.2f} bps/day (t={t:+.2f})")

specified book (long calm / short distressed):
   1 bp one-way: gross -4.35 -> net -6.48 bps/day (cost 2.14/day, t=-3.49)
  5 bps one-way: gross -4.35 -> net -14.48 bps/day (cost 10.14/day, t=-7.79)
sign-flipped REBOUND book (long distressed / short calm):
   1 bp one-way: net +2.21 bps/day (t=+1.19)
  5 bps one-way: net -5.79 bps/day (t=-3.11)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted distress relation (deep drawdown -> low forward return -> positive calm-minus-distressed spread).

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from max_drawdown import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=813+s, n_assets=40, n_days=1500))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.004, seed=813, n_assets=40, n_days=1500))
print(f"planted (edge=0.004): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.47 (sd 1.08), |t|>=2 in 1/8


planted (edge=0.004): NW t = +9.03, Welch t = +9.18


## Verdict

- **Signal — Weak.** Sorting on the trailing 12-month maximum drawdown, the long-calm / short-distressed spread is **-4.35 bps/day** (NW *t* = **-2.36**): the *distressed* names **out-earned** — a **drawdown reversal**, one of the two outcomes the claim entertained. The 1,000-permutation placebo confirms it isn't a lucky sort (~4.2σ into the left tail). But it is **not robust across eras** (*t* = -1.10 in 2010–2017 vs -2.09 in 2018–2026), so it clears the pooled |t|≥2 bar only marginally and on one half of the sample — **Weak, not Real**. The 20-seed synthetic control recovers a *planted distress* relation cleanly (*t* = +9.03) and is quiet on the null. Survivorship biases the magnitude (the deepest drawdowns — permanent losers — are absent).
- **Tradability — Mirage.** The specified book loses money (**-6.48 bps/day** net at 1 bp). The profitable *rebound* direction earns only **+2.21 bps/day** net at a fantasy 1 bp (*t* = +1.19 — not significant) and turns negative (-5.79) by 5 bps. No paycheck either way.